# Phase 0: Sanity Check — 기본 채널 추정 동작 검증

**목표**: ReEsNet 기반 채널 추정이 우리 Sionna RT 데이터에서 논문 수준으로 동작하는지 확인

## 검증 항목
1. 데이터 로드 및 형상 확인
2. 단일 BS에서 PlainEstimator (ReEsNet) 학습
3. SNR별 NMSE 곡선
4. LS / LMMSE baseline 대비 성능

## 성공 기준
- NMSE < **-15 dB** @ SNR=20 dB (ReEsNet 논문 참고)
- DNN > LMMSE > LS 순서가 유지되어야 함

In [ ]:
import sys
sys.path.insert(0, '../..')

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from src.config import SceneConfig, DatasetConfig
from src.data.dataset import ChannelEstimationDataset
from src.data.utils import nmse, nmse_db, complex_to_real, add_awgn
from src.models.baselines import PlainEstimator, LMMSEEstimator, LSEstimator
from src.training.trainer import train_local, evaluate, evaluate_per_snr

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

## 0-1. 데이터 확인

In [ ]:
DATA_DIR = '../../data/channels'

# Load metadata
meta = pd.read_parquet(f'{DATA_DIR}/metadata.parquet')
print(f'Total samples: {len(meta)}')
print(f'Snapshots: {meta.snapshot_id.nunique()}')
print(f'UEs per snapshot: ~{len(meta) // meta.snapshot_id.nunique()}')
print(f'\nSamples per BS:')
print(meta.groupby('bs_id').size())

# Check channel shapes
snap0 = np.load(f'{DATA_DIR}/snapshot_0000/channels.npz')
print(f'\nCFR shape: {snap0["cfr"].shape}')  # (N_UE, n_rx_ant, n_tx_ant, n_sc)
print(f'CIR a shape: {snap0["cir_a"].shape}')
print(f'CIR tau shape: {snap0["cir_tau"].shape}')

In [ ]:
# Visualize sample channels
cfr = snap0['cfr']  # (N_UE, n_rx, n_tx, n_sc) complex
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for i in range(min(8, cfr.shape[0])):
    ax = axes[i // 4, i % 4]
    h = cfr[i, 0, 0, :]  # first rx_ant, first tx_ant
    ax.plot(20*np.log10(np.abs(h) + 1e-10))
    ax.set_title(f'UE {i} (BS {meta.iloc[i]["bs_id"]})')
    ax.set_xlabel('Subcarrier')
    ax.set_ylabel('|H| (dB)')
    ax.grid(True, alpha=0.3)
plt.suptitle('Sample CFR Magnitude (dB)', fontsize=14)
plt.tight_layout()
plt.show()

## 0-2. 단일 BS 학습 (PlainEstimator = ReEsNet)

In [ ]:
# Pick BS with most data
bs_counts = meta.groupby('bs_id').size()
target_bs = int(bs_counts.idxmax())
print(f'Training on BS{target_bs} ({bs_counts[target_bs]} samples)')

# Dataset: fixed SNR=20 dB for initial test
TRAIN_SNR = 20.0
BATCH_SIZE = 32

ds = ChannelEstimationDataset(
    data_dir=DATA_DIR, bs_ids=[target_bs],
    snr_range_db=(TRAIN_SNR, TRAIN_SNR),  # Fixed SNR for clean evaluation
)
n = len(ds)
n_val = int(n * 0.2)
n_train = n - n_val
train_ds, val_ds = torch.utils.data.random_split(ds, [n_train, n_val])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

print(f'Train: {n_train}, Val: {n_val}')

# Quick shape check
batch = next(iter(train_loader))
print(f'Input shape: {batch["input"].shape}')   # (B, 2, 8, 1024)
print(f'Target shape: {batch["target"].shape}')  # (B, 2, 8, 1024)

In [ ]:
# Train PlainEstimator (ReEsNet baseline)
model = PlainEstimator(
    encoder_channels=64, encoder_blocks=3,
    task_head_blocks=2, kernel_size=3,
).to(device)

print(f'Model params: {sum(p.numel() for p in model.parameters()):,}')

result = train_local(
    model, train_loader, val_loader,
    epochs=100, lr=1e-3, patience=20, device=device,
)

best_nmse_db = 10 * np.log10(result['best_val'])
print(f'\nBest val NMSE: {best_nmse_db:.2f} dB (epoch {result["best_epoch"]})')
print(f'Target: < -15 dB')
print(f'Result: {"PASS ✓" if best_nmse_db < -15 else "FAIL ✗"}')

In [ ]:
# Training curve
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(10*np.log10(result['train_losses']), label='Train', alpha=0.7)
ax.plot(10*np.log10(result['val_losses']), label='Val', linewidth=2)
ax.axhline(y=-15, color='r', linestyle='--', label='Target: -15 dB')
ax.set_xlabel('Epoch')
ax.set_ylabel('NMSE (dB)')
ax.set_title(f'ReEsNet Training on BS{target_bs} @ SNR={TRAIN_SNR} dB')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('phase0_training_curve.png', dpi=150)
plt.show()

## 0-3. SNR별 NMSE 곡선 + Baseline 비교

In [ ]:
# Now train with random SNR range for more robust model
ds_mixed = ChannelEstimationDataset(
    data_dir=DATA_DIR, bs_ids=[target_bs],
    snr_range_db=(0.0, 30.0),
)
n2 = len(ds_mixed)
n2_val = int(n2 * 0.2)
train2, val2 = torch.utils.data.random_split(ds_mixed, [n2 - n2_val, n2_val])
train_loader2 = DataLoader(train2, batch_size=BATCH_SIZE, shuffle=True)
val_loader2 = DataLoader(val2, batch_size=BATCH_SIZE)

model_mixed = PlainEstimator(encoder_channels=64, encoder_blocks=3).to(device)
train_local(model_mixed, train_loader2, val_loader2, epochs=100, device=device)

In [ ]:
# LMMSE baseline
lmmse = LMMSEEstimator(n_ant_pairs=8, n_subcarriers=1024).to(device)
train_local(lmmse, train_loader2, val_loader2, epochs=50, lr=1e-2, device=device)

In [ ]:
# Evaluate at specific SNR values
SNR_LIST = [0, 5, 10, 15, 20, 25, 30]

results_per_snr = {}
for name, mdl in [('ReEsNet', model_mixed), ('LMMSE', lmmse)]:
    results_per_snr[name] = evaluate_per_snr(
        mdl, ChannelEstimationDataset, DATA_DIR, [target_bs],
        SNR_LIST, batch_size=64, device=device,
    )

# LS baseline (identity — just returns noisy input)
ls_results = {}
for snr in SNR_LIST:
    ds_snr = ChannelEstimationDataset(data_dir=DATA_DIR, bs_ids=[target_bs], fixed_snr_db=snr)
    loader = DataLoader(ds_snr, batch_size=64)
    total = 0
    n = 0
    for batch in loader:
        x = batch['input']
        y = batch['target']
        total += nmse(x, y).item() * x.size(0)  # LS = just return input
        n += x.size(0)
    ls_results[snr] = 10 * torch.log10(torch.tensor(total / n)).item()
results_per_snr['LS'] = ls_results

print('\nSNR (dB) | LS (dB) | LMMSE (dB) | ReEsNet (dB)')
print('-' * 50)
for snr in SNR_LIST:
    print(f'  {snr:5.0f}   | {results_per_snr["LS"][snr]:7.2f} | '
          f'{results_per_snr["LMMSE"][snr]:10.2f} | {results_per_snr["ReEsNet"][snr]:11.2f}')

In [ ]:
# Plot: SNR vs NMSE
fig, ax = plt.subplots(figsize=(8, 5))
markers = {'LS': 's', 'LMMSE': '^', 'ReEsNet': 'o'}
colors = {'LS': 'gray', 'LMMSE': 'C1', 'ReEsNet': 'C0'}

for name in ['LS', 'LMMSE', 'ReEsNet']:
    vals = [results_per_snr[name][s] for s in SNR_LIST]
    ax.plot(SNR_LIST, vals, marker=markers[name], label=name,
            color=colors[name], linewidth=2, markersize=8)

ax.set_xlabel('SNR (dB)', fontsize=12)
ax.set_ylabel('NMSE (dB)', fontsize=12)
ax.set_title(f'Channel Estimation Performance — BS{target_bs}', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('phase0_snr_nmse.png', dpi=150)
plt.show()

# Validation
reesnet_20 = results_per_snr['ReEsNet'][20]
print(f'\n=== Phase 0 결과 ===')
print(f'ReEsNet @ SNR=20 dB: {reesnet_20:.2f} dB')
print(f'LS > LMMSE > ReEsNet 순서: {results_per_snr["LS"][20] > results_per_snr["LMMSE"][20] > reesnet_20}')
print(f'NMSE < -15 dB: {reesnet_20 < -15}')
print(f'\n전체 판정: {"PASS ✓ → Phase 1 진행" if reesnet_20 < -15 else "FAIL ✗ → 원인 분석 필요"}')

## 0-4. 다중 BS 독립 학습

In [ ]:
# Train on each BS independently
all_bs = sorted(meta.bs_id.unique())
bs_results = {}

for bs_id in all_bs:
    n_bs = int(bs_counts[bs_id])
    if n_bs < 10:
        print(f'BS{bs_id}: only {n_bs} samples, skipping')
        continue
    
    print(f'\nBS{bs_id} ({n_bs} samples)...')
    ds_bs = ChannelEstimationDataset(data_dir=DATA_DIR, bs_ids=[bs_id], snr_range_db=(0, 30))
    n_v = max(int(len(ds_bs) * 0.2), 1)
    n_t = len(ds_bs) - n_v
    t_ds, v_ds = torch.utils.data.random_split(ds_bs, [n_t, n_v])
    
    m = PlainEstimator(encoder_channels=64, encoder_blocks=3).to(device)
    res = train_local(
        m, DataLoader(t_ds, batch_size=BATCH_SIZE, shuffle=True),
        DataLoader(v_ds, batch_size=BATCH_SIZE),
        epochs=80, device=device, verbose=False,
    )
    bs_results[bs_id] = 10 * np.log10(res['best_val'])
    print(f'  NMSE: {bs_results[bs_id]:.2f} dB')

print('\n=== 다중 BS 결과 ===')
for bs, db in sorted(bs_results.items()):
    status = '✓' if db < -10 else '✗'
    print(f'  BS{bs}: {db:.2f} dB [{status}]')

In [ ]:
# Bar chart of per-BS performance
fig, ax = plt.subplots(figsize=(10, 5))
bs_ids = sorted(bs_results.keys())
vals = [bs_results[b] for b in bs_ids]
colors = ['green' if v < -10 else 'red' for v in vals]
ax.bar(range(len(bs_ids)), vals, color=colors, alpha=0.7)
ax.set_xticks(range(len(bs_ids)))
ax.set_xticklabels([f'BS{b}\n({int(bs_counts[b])})' for b in bs_ids])
ax.axhline(y=-10, color='r', linestyle='--', label='Target: -10 dB')
ax.set_xlabel('BS ID (sample count)')
ax.set_ylabel('NMSE (dB)')
ax.set_title('Independent Training per BS (SNR range 0-30 dB)')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('phase0_per_bs.png', dpi=150)
plt.show()